In [ ]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad
import joblib

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
# ... restantes dos imports ...

# =====================================================
# Salva tudo o que aparece no terminal em um arquivo TXT
# =====================================================

log = open("saida_terminal.txt", "w", encoding="utf-8", buffering=1)

class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for s in self.streams:
            s.write(data)
            s.flush()

    def flush(self):
        for s in self.streams:
            s.flush()

sys.stdout = Tee(sys.__stdout__, log)
sys.stderr = Tee(sys.__stderr__, log)

In [ ]:
# Load experimental data
atlas_data = pd.read_csv('../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [ ]:

b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

n_points = 10000



# ensemble_parameters = {
#     'atlas': {
#         'pl': {
#             'epsilon': 0.0753,
#             'mg': 0.421,
#             'a1': 1.517,
#             'a2': 2.05
#         }
#     }
# }

# ensemble_atlas = 'atlas'  
# ensemble_totem = 'totem'

# log_model_type = 'log'
# pl_model_type = 'pl'   

# def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=0.9, upper_factor=1.1):
#     # Obtém os parâmetros iniciais
#     initial_params = ensemble_parameters[ensemble_name][model_type]
    
#     # Cria as variações
#     initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
#     initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
#     return initial_params, initial_params_low, initial_params_high

# # Get parameters for selected configuration
# initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# # Para Atlas
# initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
#     get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [ ]:
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323

def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):

    def integrand(y, x, mg, a1, a2, m2_func, q_val):
        k        = sqrt_s * x
        phi      = 2*np.pi*y
        jacobian = 2*np.pi*sqrt_s
        return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
                  T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian
    def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func, q_val),
                0, 1, n=n_points
            )[0]

    integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
    
    return integral_value

In [ ]:
contador = 0 
t0 = time.perf_counter()
t_anterior = t0

def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }

    # ============================
    # apenas para debug, imprime os parâmetros atuais
    # ============================
    global contador, t_anterior

    contador += 1

    if contador % 100 == 0:

        agora = time.perf_counter()

        print("\n=============================")
        print(f"Iteração : {contador}")
        print(f"Tempo total : {agora-t0:.2f} s")
        print(f"Últimas 100 chamadas : {agora-t_anterior:.2f} s")
        print("=============================")
        print(f"eps = {eps:.6f}")
        print(f"mg  = {mg:.6f}")
        print(f"a1  = {a1:.6f}")
        print(f"a2  = {a2:.6f}")
        print("\n")

        t_anterior = agora
    # ============================



    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [ ]:
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='log')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='log')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='log')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)


chi2_total = chi2_7 + chi2_8 + chi2_13


In [ ]:
minuit_born = Minuit(
    chi2_total,
    mg = 0.356,
    a1 = 1.373,
    a2 = 2.5,
    eps = 0.0753)

minuit_born.print_level = 2

minuit_born.tol = 1e-8
minuit_born.strategy = 2

minuit_born.limits['mg'] = (0.2, 0.5)
minuit_born.limits['a1'] = (0.5, 2.0)
minuit_born.limits['a2'] = (2.0, 4.0)
minuit_born.limits['eps'] = (0.001, 0.15)


minuit_born.errors["mg"]  = 0.010
minuit_born.errors["eps"] = 0.001
minuit_born.errors["a1"]  = 0.020
minuit_born.errors["a2"]  = 0.100


In [ ]:

ncall = 5000

minuit_born.simplex(ncall = ncall)
minuit_born.simplex(ncall = ncall)
minuit_born.simplex(ncall = ncall)



minuit_born.migrad(ncall = ncall)
minuit_born.migrad(ncall = ncall)
minuit_born.migrad(ncall = ncall)

minuit_born.hesse()



In [ ]:
print(f"chi2 = {minuit_born.fval}")
print(f"ndof = {len(y_7_atlas) + len(y_8_atlas) + len(y_13_atlas) - minuit_born.nfit}")

In [ ]:
# Restaurar as saídas originais
sys.stdout = sys.__stdout__
sys.stderr = sys.__stderr__

# Agora fechar o arquivo
log.close()

print("Log salvo em saida_terminal.txt")